# Validate the generated process dataset

Unlike the TEP workflow, the CausalAI generator already writes Parquet and CSV files, so no raw-data conversion step is needed. This notebook checks the generated artifacts, sample labels, sampling interval, and per-run relationship changes before preprocessing.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: <repo>


In [2]:
import json

import numpy as np
import pandas as pd

from carl.causalai_process import FAULTS, VARIABLE_NAMES


DATA_DIR = PROJECT_ROOT / "data" / "processed" / "causalAIProcess"
required_files = [
    "traces.parquet",
    "runs.csv",
    "causal_graph.csv",
    "fault_coefficients.csv",
    "metadata.json",
]
missing_files = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing generated dataset files in {DATA_DIR}: {missing_files}")

traces = pd.read_parquet(DATA_DIR / "traces.parquet")
runs = pd.read_csv(DATA_DIR / "runs.csv")
causal_graph = pd.read_csv(DATA_DIR / "causal_graph.csv")
fault_coefficients = pd.read_csv(DATA_DIR / "fault_coefficients.csv")
metadata = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))

expected_run_count = (1 + len(FAULTS)) * metadata["runs_per_class"]
expected_samples = expected_run_count * metadata["samples_per_run"]
assert len(runs) == expected_run_count
assert len(traces) == expected_samples
assert set(runs["fault_type"]) == {"healthy", *FAULTS}
fault_runs = runs.loc[runs["fault_type"] != "healthy"]
severity_min, severity_max = metadata["fault_severity_range"]
assert fault_runs["fault_severity"].between(severity_min, severity_max).all()
assert set(traces["run_id"]) == set(runs["run_id"])
assert set(VARIABLE_NAMES).issubset(traces.columns)
assert np.isfinite(traces[list(VARIABLE_NAMES)].to_numpy()).all()
assert (traces["feed_concentration"] > 0).all()
assert (traces["reactor_concentration"] > 0).all()
assert traces["product_quality"].between(0, 1, inclusive="neither").all()

for run in runs.itertuples():
    trajectory = traces.loc[traces["run_id"] == run.run_id].sort_values("time_step")
    np.testing.assert_array_equal(
        trajectory["time_step"].to_numpy(),
        np.arange(metadata["samples_per_run"]),
    )
    assert trajectory["time_seconds"].diff().dropna().eq(
        metadata["sampling_interval_seconds"]
    ).all()
    expected_onset = (
        -1
        if run.fault_type == "healthy"
        else metadata["samples_per_run"] // 5
    )
    assert run.fault_start_step == expected_onset
    expected_labels = (
        np.zeros(metadata["samples_per_run"], dtype=int)
        if expected_onset < 0
        else (np.arange(metadata["samples_per_run"]) >= expected_onset).astype(int)
    )
    np.testing.assert_array_equal(trajectory["anomaly"].to_numpy(), expected_labels)

edge_columns = ["source", "target", "lag_steps"]
baseline_edges = set(map(tuple, causal_graph[edge_columns].to_numpy()))
assert len(fault_coefficients) == (runs["fault_type"] != "healthy").sum() * len(causal_graph)
for run_id, run_changes in fault_coefficients.groupby("run_id"):
    assert set(map(tuple, run_changes[edge_columns].to_numpy())) == baseline_edges
    assert (run_changes["coefficient_before"] != run_changes["coefficient_after"]).all()

print(f"Validated {len(runs)} runs and {len(traces):,} samples.")
print(f"Sampling interval: {metadata['sampling_interval_seconds']} second(s)")
display(runs.groupby(["fault_type", "split"]).size().unstack(fill_value=0))
display(causal_graph)

Validated 2000 runs and 2,000,000 samples.
Sampling interval: 1.0 second(s)


split,test,train,validation
fault_type,,,
healthy,75,350,75
hydraulic_coupling_shift,75,350,75
reaction_coupling_shift,75,350,75
thermal_coupling_shift,75,350,75


,source,target,lag_steps,coefficient,function,mechanism
0,feed_flow,feed_flow,1,0.75,identity,input_persistence
1,feed_concentration,feed_concentration,1,0.70,identity,input_persistence
2,coolant_flow,coolant_flow,1,0.75,identity,input_persistence
3,reactor_concentration,reactor_concentration,1,0.62,identity,state_persistence
4,feed_concentration,reactor_concentration,1,0.22,tanh,feed_mixing
5,feed_flow,reactor_concentration,1,-0.12,tanh,hydraulic_residence
6,reactor_temperature,reactor_concentration,1,0.08,tanh,reaction_kinetics
7,reactor_temperature,reactor_temperature,1,0.64,identity,state_persistence
8,feed_flow,reactor_temperature,1,0.08,tanh,hydraulic_residence
9,reactor_concentration,reactor_temperature,1,0.20,tanh,reaction_heat
